In [ ]:
import requests
import json
import sqlite3

#API endpoit
url = "https://civicdb.org/api/graphql"
headers = {
    "Content-Type": "application/json",
}


## Fetch Variants

In [ ]:

query = """
query browseVariants($after: String) {
  variants(first: 300, after: $after) {
    nodes {
          id
          name
          feature {
            id
          }
          molecularProfiles {
              nodes {
                  id
                  name
                  description
                  evidenceItems {
                      nodes {
                          id
                          name
                          disease {
                              id
                              name
                          }
                      }
                  }
              }
          }
      }
    pageInfo {
      endCursor
      hasNextPage
    }
    totalCount
  }
}
"""

all_variants = []
variables = {"after": None}

while True:
    response = requests.post(url, json={'query': query, 'variables': variables}, headers=headers)
    response_json = response.json()
    
    if 'data' in response_json:
        variants = response_json["data"]["variants"]["nodes"]
        all_variants.extend(variants)
        
        page_info = response_json["data"]["variants"]["pageInfo"]
        if not page_info["hasNextPage"]:
            break
        variables["after"] = page_info["endCursor"]
    else:
        print("Error in response:", response_json.get('errors'))
        break

print(f"Total profiles fetched: {len(all_variants)}")

In [ ]:
#list of variations to exclude
VARIATIONS_TO_EXCLUDE = [
    "activation",
    "allele",
    "alu",
    "alteration",
    "alternative",
    "amplification",
    "and",
    "conserve",
    "deficient",
    "deletion",
    "depletion",
    "demethylation",
    "domain",
    "double",
    "duplication",
    "expression",
    "fas",
    "function",
    "fusion",
    "gain",
    "inactivation",
    "increase",
    "insert",
    "knockdown",
    "loss",
    "local",
    "methylation",
    "mut",
    "overexpression",
    "phosphorylation",
    "polymorphisme",
    "positive",
    "promoter",
    "rearrangement",
    "repeat",
    "transcripts",
    "translocation",
    "underexpression",
    "upregulation",
    "shift",
    "variant",
    "variation",
    "wild",
    "P772_H773insH", 
    "nm0", 
    "fas"]


filtered_variants = [
    variant for variant in all_variants
    if not any(exclusion in variant['name'].lower() for exclusion in VARIATIONS_TO_EXCLUDE)
]

print(f"Total filtered variants: {len(filtered_variants)}")

In [ ]:
filtered_variants

In [ ]:
import csv

# Specify the output CSV file path
output_file = "filtered_variants_names.csv"

# Extract the "name" fields from filtered_variants
variant_names = [variant['name'] for variant in filtered_variants]

# Write the names to the CSV file
with open(output_file, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow(["Name"])  # Write the header
    for name in variant_names:
        writer.writerow([name])

print(f"Filtered variant names have been saved to {output_file}")

# store Variants in SQL db

In [ ]:
#store variants

# Connect to the SQLite database (or create it if it doesn't exist)
conn = sqlite3.connect('../database.db')

with open('../genomics.sql') as f:
        conn.executescript(f.read())

cursor = conn.cursor()
# Insert the variants into the table
for variant in filtered_variants:
    mp_ids      = []
    disease_ids = []

    for profile in variant["molecularProfiles"]["nodes"]:
        mp_ids.append(profile["id"])
        for evidenceItem in profile["evidenceItems"]["nodes"]:
            if evidenceItem["disease"] and evidenceItem["disease"]["id"]:
                disease_ids.append(evidenceItem["disease"]["id"])
    
    disease_ids = json.dumps(list(set(disease_ids)))  # Remove duplicates and convert to JSON format
    mp_ids = json.dumps(list(set(mp_ids)))  # Remove duplicates and convert to JSON format
    cursor.execute('''
    INSERT OR REPLACE INTO variants (id, name, gene_id, molecular_profiles, diseases, db_source)
    VALUES (?, ?, ?, ?, ?, ?)
    ''', (variant['id'], variant['name'], variant["feature"]["id"], mp_ids, disease_ids, "civic"))

# Commit the transaction and close the connection
conn.commit()
conn.close()

## Fetch Genes


In [ ]:
query = """
query browseGenes($after: String) {
    genes(first: 300, after: $after) {
        nodes {
            id
            name
            description
            variants {
                nodes {
                    id
                    name
                    molecularProfiles {
                        nodes {
                            id
                            name
                            description
                            evidenceItems {
                                nodes {
                                    id
                                    name
                                    disease {
                                        id
                                        name
                                    }
                                }
                            }
                        }
                    }
                }
            }
        }
        pageInfo {
            endCursor
            hasNextPage
        }
        totalCount
    }
}
"""


all_genes = []
variables = {"after": None}

while True:
    response = requests.post(url, json={'query': query, 'variables': variables}, headers=headers)
    response_json = response.json()
    
    if 'data' in response_json:
        genes = response_json["data"]["genes"]["nodes"]
        all_genes.extend(genes)
        
        page_info = response_json["data"]["genes"]["pageInfo"]
        if not page_info["hasNextPage"]:
            break
        variables["after"] = page_info["endCursor"]
    else:
        print("Error in response:", response_json.get('errors'))
        break

print(f"Total genes fetched: {len(all_genes)}")

In [ ]:
all_genes

In [ ]:
#checking the disease content (mostly empty)
i = 0
for gene in all_genes:
    for variant in gene["variants"]["nodes"]:
        for profile in variant["molecularProfiles"]["nodes"]:
            if profile["evidenceItems"]["nodes"] and profile["evidenceItems"]["nodes"][0]["disease"]:
                    i = i+1
                    print(f"Gene: {gene['name']}, Profile: {profile['name']}, Disease: {profile['evidenceItems']['nodes'][0]['disease']}")
print(i)

##### store genes in SQL db

In [ ]:
#store genes

# Connect to the SQLite database (or create it if it doesn't exist)
conn = sqlite3.connect('../database.db')

with open('../genomics.sql') as f:
        conn.executescript(f.read())

cursor = conn.cursor()
# Insert the genes into the table
for node in all_genes:
    disease_ids = []
    variant_ids = []
    mp_ids      = []
    for variant in node["variants"]["nodes"]:
        variant_ids.append(variant["id"])
        for profile in variant["molecularProfiles"]["nodes"]:
            mp_ids.append(profile["id"])
            for evidenceItem in profile["evidenceItems"]["nodes"]:
                if evidenceItem["disease"] and evidenceItem["disease"]["id"]:
                    disease_ids.append(evidenceItem["disease"]["id"])
    disease_ids = json.dumps(list(set(disease_ids)))  # Remove duplicates and convert to JSON format
    variant_ids = json.dumps(list(set(variant_ids)))  # Remove duplicates and convert to JSON format
    mp_ids = json.dumps(list(set(mp_ids)))  # Remove duplicates and convert to JSON format

    cursor.execute('''
    INSERT OR REPLACE INTO genes (id, name, description, molecular_profiles, variants, diseases, db_source)
    VALUES (?, ?, ?, ?, ?, ?, ?)
    ''', (node['id'], node['name'], node['description'], mp_ids, variant_ids, disease_ids, "civic"))

# Commit the transaction and close the connection
conn.commit()
conn.close()

# Fetch diseases

In [ ]:
query = """
query browseDiseases($after: String) {
  diseases(first: 300, after: $after) {
    nodes {
        id
        name
    }  
    pageInfo {
      endCursor
      hasNextPage
    }
    totalCount
  }
}
"""

all_diseases = []
variables = {"after": None}

while True:
    response = requests.post(url, json={'query': query, 'variables': variables}, headers=headers)
    response_json = response.json()
    
    if 'data' in response_json:
        diseases = response_json["data"]["diseases"]["nodes"]
        all_diseases.extend(diseases)
        
        page_info = response_json["data"]["diseases"]["pageInfo"]
        if not page_info["hasNextPage"]:
            break
        variables["after"] = page_info["endCursor"]
    else:
        print("Error in response:", response_json.get('errors'))
        break

print(f"Total profiles fetched: {len(all_diseases)}")

#### store diseases in SQL db

In [ ]:
# Connect to the SQLite database (or create it if it doesn't exist)
conn = sqlite3.connect('../database.db')

with open('../genomics.sql') as f:
        conn.executescript(f.read())

cursor = conn.cursor()
# Insert the filtered molecular profiles into the table
for disease in all_diseases:
    cursor.execute('''
    INSERT OR REPLACE INTO diseases (id, name, db_source)
    VALUES (?, ?, ?)
    ''', (disease['id'], disease['name'], "civic"))

# Commit the transaction and close the connection
conn.commit()
conn.close()

# Fetch Molecular Profiles

In [ ]:
query = """
query browseMolecularProfiles($after: String) {
  molecularProfiles(first: 300, after: $after) {
    edges {
      node {
        id
        name
        description
        molecularProfileScore
        variants {
          id
          name
          feature {
            id
            name
          }
        }
        assertions {
          nodes{
            id
            name
            description
            disease{
              id
              name
            } 
          }
        } 
      }
    }
    pageInfo {
      endCursor
      hasNextPage
    }
    totalCount
  }
}
"""

all_molecular_profiles = []
variables = {"after": None}

while True:
    response = requests.post(url, json={'query': query, 'variables': variables}, headers=headers)
    response_json = response.json()
    
    if 'data' in response_json:
        molecular_profiles = response_json["data"]["molecularProfiles"]["edges"]
        all_molecular_profiles.extend(molecular_profiles)
        
        page_info = response_json["data"]["molecularProfiles"]["pageInfo"]
        if not page_info["hasNextPage"]:
            break
        variables["after"] = page_info["endCursor"]
    else:
        print("Error in response:", response_json.get('errors'))
        break

print(f"Total profiles fetched: {len(all_molecular_profiles)}")

##### filter MP

In [ ]:
#filter out MP with score 0
molecular_profiles_filtered = [edge for edge in all_molecular_profiles if edge["node"]["molecularProfileScore"] != 0]

molecular_profiles_filtered = [
    mp for mp in molecular_profiles_filtered
    if not any(exclusion in mp["node"]['name'].lower() for exclusion in VARIATIONS_TO_EXCLUDE)
]

print(f"Total filtered profiles: {len(molecular_profiles_filtered)}")

In [ ]:
molecular_profiles_filtered

##### Store MP in sql table

In [ ]:

# Connect to the SQLite database (or create it if it doesn't exist)
conn = sqlite3.connect('../database.db')

with open('../genomics.sql') as f:
        conn.executescript(f.read())

cursor = conn.cursor()
# Insert the filtered molecular profiles into the table
for profile in molecular_profiles_filtered:
    disease_ids = []
    variant_ids = []
    node = profile['node']
    for variant in node["variants"]:
        variant_ids.append(variant["id"])
    
    for assertion in node["assertions"]["nodes"]:
        if assertion["disease"] and assertion["disease"]["id"]:
            disease_ids.append(assertion["disease"]["id"])
    disease_ids = json.dumps(list(set(disease_ids)))  # Remove duplicates
    variant_ids = json.dumps(list(set(variant_ids)))  # Remove duplicates and convert to JSON format
    cursor.execute('''
    INSERT OR REPLACE INTO molecular_profiles (id, name, description, variants, disease, molecularProfileScore, db_source)
    VALUES (?, ?, ?, ?, ?, ?, ?)
    ''', (node['id'], node['name'], node['description'], variant_ids, disease_ids,  node['molecularProfileScore'], "civic"))

# Commit the transaction and close the connection
conn.commit()
conn.close()

## Test Database

In [ ]:
#test db
# Connect to the SQLite database
conn = sqlite3.connect('../database.db')
cursor = conn.cursor()

# Execute a query to retrieve all data from the molecular_profiles table
cursor.execute("SELECT * FROM genes")

# Fetch all rows from the executed query
rows = cursor.fetchall()

# Display the data
for gene in rows:
    print(gene)

# Close the connection
conn.close()

In [ ]:
gene[3]

In [ ]:
#check disease content
# Connect to the SQLite database
conn = sqlite3.connect('../database.db')
cursor = conn.cursor()

# Execute a query to retrieve all data from the molecular_profiles table
cursor.execute("SELECT * FROM genes")

# Fetch all rows from the executed query
rows = cursor.fetchall()

# Display the data
i = 0
for gene in rows:
    if len(gene[5])>2:
        print(gene[5])
        i = i+1

# Close the connection
conn.close()
print(i)

In [ ]:
gene[5]

In [ ]:
#checking the assertions content (mostly empty)
i = 0
for gene in all_genes:
    for variant in gene["variants"]["nodes"]:
        for profile in variant["molecularProfiles"]["nodes"]:
            if profile["assertions"]["nodes"]:  # If assertions exist
                i = i+1
                print(f"Gene: {gene['name']}, Profile: {profile['name']}, Assertions: {profile['assertions']['nodes']}")
print(i)

In [ ]:
#test retrieval by name
def get_all_molecular_profiles(db_path):
    # Connect to the SQLite database
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Execute a query to retrieve all data from the molecular_profiles table
    cursor.execute("SELECT * FROM molecular_profiles")

    # Fetch all rows from the executed query
    rows = cursor.fetchall()

    # Close the connection
    conn.close()

    return rows

NAME = "BRAC2 Mutation"

print(rows)
    
    #if profile['node']['name'] == NAME:
    #     print(profile['node'])


In [ ]:
def get_all_molecular_profiles_with_keys(db_path):
    # Connect to the SQLite database
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Execute a query to retrieve all data from the molecular_profiles table
    cursor.execute("SELECT * FROM molecular_profiles")

    # Fetch all rows from the executed query
    rows = cursor.fetchall()

    # Get the column names from the cursor description
    column_names = [description[0] for description in cursor.description]

    # Close the connection
    conn.close()

    # Combine column names with rows
    profiles_with_keys = [dict(zip(column_names, row)) for row in rows]

    return profiles_with_keys

# Use the function to get the profiles
profiles_with_keys = get_all_molecular_profiles_with_keys('../database.db')
print(profiles_with_keys)

In [ ]:
profiles_with_keys

In [ ]:
for profile in profiles_with_keys:
    print(profile['disease'])